Hi! We will predict wine quality. Firstly i will do data visualisation and data exploring then predict the quality of wine using cross validation! Let's get started!


![](https://machinelearninghd.com/wp-content/uploads/2021/03/wine-quality.jpg)

**Table of contents of this notebook:**

1. [Importing Necessary Libraries](#1)

2. [Loading The Data](#2)

3. [Exploratory Data Analysis](#3)

4. [Feature Engineering](#4)

5. [Data Preprocessing](#5)

6. [Models](#6)

<h1  style="text-align: center" class="list-group-item list-group-item-action active">1. Importing Necessary Libraries</h1><a id = "1"></a>

In [ ]:
pip install feature-engine

In [ ]:
#Importing required packages.
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from feature_engine.outliers import Winsorizer
import feature_engine.transformation as vt

# for one hot encoding with sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier

from collections import Counter

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
%matplotlib inline

# Ignoring Unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

<h1  style="text-align: center" class="list-group-item list-group-item-action active">2.Loading The Data</h1><a id = "2"></a>

In [ ]:
df = pd.read_csv('../input/red-wine-quality-cortez-et-al-2009/winequality-red.csv')

In [ ]:
df.head(10)

In [ ]:
df.shape

We have 1599 observations and 12 feature columns `quailty` feature will be our target value.
* Out of which one is dependent variable and rest 11 are independent variables

In [ ]:
df.info()

All of our features are numerical. 

<h1  style="text-align: center" class="list-group-item list-group-item-action active">3. Exploratory Data Analysis</h1><a id = "3"></a>

Let's start with seeing the ratio of the variables by pie plot.

In [ ]:
df.quality.unique() #we have 3-9 unique values. 

In [ ]:
plt.figure(1, figsize=(10,10))
df['quality'].value_counts().plot.pie(autopct="%1.1f%%")
plt.show()

In [ ]:
df.quality.value_counts(ascending=False)

In [ ]:
# Function to create a histogram, and a boxplot and scatter plot.
def diagnostic_plots(df, variable,target):
    # The function takes a dataframe (df) and
    # the variable of interest as arguments.

    # Define figure size.
    plt.figure(figsize=(20, 4))

    # histogram
    plt.subplot(1, 4, 1)
    sns.histplot(df[variable], bins=30,color = 'r')
    plt.title('Histogram')


    # scatterplot
    plt.subplot(1, 4, 2)
    plt.scatter(df[variable],df[target],color = 'g')
    plt.title('Scatterplot')
    
    
    # boxplot
    plt.subplot(1, 4, 3)
    sns.boxplot(y=df[variable],color = 'b')
    plt.title('Boxplot')
    
    # barplot
    plt.subplot(1, 4, 4)
    sns.barplot(x = target, y = variable, data = df)   
    plt.title('Barplot')
    
    
    plt.show()

In [ ]:
for variable in df:
    diagnostic_plots(df,variable,'quality')

We can see the skewness of the variables and outliers pretty much in every feature.

* “pH” column appears to be normally distributed.

* Remaining all independent variables are right skewed/positively skewed.

* In our data set except “alcohol” all other features columns shows outliers.


![](https://i.imgflip.com/1c67ry.jpg)

Let's check the correlation of the features.

In [ ]:
corr = df.corr()
plt.figure(figsize=(20, 9))
k = 12 #number of variables for heatmap
cols = corr.nlargest(k, 'quality')['quality'].index
cm = np.corrcoef(df[cols].values.T)
sns.set(font_scale=1.25)
hm = sns.heatmap(cm, cbar=True, annot=True, square=True, fmt='.2f', annot_kws={'size': 10}, yticklabels=cols.values, xticklabels=cols.values,cmap="Blues")
plt.show()

Dark shades represents positive correlation while lighter shades represents negative correlation.

Observations:
* Here we can infer that “density” has strong positive correlation with “fixed acidity” whereas it has strong negative correlation with “alcohol”.

* “free sulphur dioxide” and “pH”, "residual sugar" has almost no correlation with “quality”.

* Since correlation is zero we can infer there is no linear relationship between these two predictors.

* Alcohol has the highest positive correlation with wine quality, followed by the various other variables such as acidity, sulphates, density & chlorides.

Let's check if our data has missing values!

In [ ]:
df.isnull().sum()

We don't have any missing values that is great!

<h1  style="text-align: center" class="list-group-item list-group-item-action active">4. Feature Engineering</h1><a id = "4"></a>
                                                                                                                

<h1  style="text-align: center" class="list-group-item list-group-item-action active">4.1 Outliers</h1>

In [ ]:
def detect_outliers(df,features):
    outlier_indices = []
    
    for c in features:
        # 1st quartile
        Q1 = np.percentile(df[c],25)
        # 3rd quartile
        Q3 = np.percentile(df[c],75)
        # IQR
        IQR = Q3 - Q1
        # Outlier step
        outlier_step = IQR * 1.5
        # detect outlier and their indeces
        outlier_list_col = df[(df[c] < Q1 - outlier_step) | (df[c] > Q3 + outlier_step)].index
        # store indeces
        outlier_indices.extend(outlier_list_col)
    
    outlier_indices = Counter(outlier_indices)
    multiple_outliers = list(i for i, v in outlier_indices.items() if v > 2)
    
    return multiple_outliers

In [ ]:
diagnostic_plots(df,'fixed acidity','quality')

In [ ]:
df.shape

In [ ]:
df.iloc[detect_outliers(df,df.columns[:-1])]

We detected some outliers! Let's cap them using `Winsorizer`.

In [ ]:
df.columns

In [ ]:
windsoriser = Winsorizer(capping_method='iqr', # choose iqr for IQR rule boundaries or gaussian for mean and std
                          tail='both', # cap left, right or both tails 
                          fold=1.5,
                          variables=['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
                                     'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
                                     'pH', 'sulphates'])

In [ ]:
diagnostic_plots(df,'fixed acidity','quality')

In [ ]:
windsoriser.fit(df)

In [ ]:
df = windsoriser.transform(df)

In [ ]:
diagnostic_plots(df,'fixed acidity','quality')

<h1  style="text-align: center" class="list-group-item list-group-item-action active">4.2 Normal Distribution</h1> 

We saw that a lot of variables show skewness. We will normalize these features using log function.

In [ ]:
#except pH
cols = ['fixed acidity', 'volatile acidity', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
       'sulphates', 'alcohol']

In [ ]:
lt = vt.LogTransformer(variables = cols)

lt.fit(df)

In [ ]:
df = lt.transform(df)

In [ ]:
#Making binary classificaion for the response variable.
#Dividing wine as good and bad by giving the limit for the quality
bins = (2, 6.5, 8)
group_names = ['bad', 'good']
df['quality'] = pd.cut(df['quality'], bins = bins, labels = group_names)

<h1  style="text-align: center" class="list-group-item list-group-item-action active">5. Data Preprocessing</h1><a id = "5"></a>

<h1  style="text-align: center" class="list-group-item list-group-item-action active">5.1 Encoding</h1>

In [ ]:
encoder = LabelEncoder()

In [ ]:
df['quality'] = encoder.fit_transform(df['quality'])

In [ ]:
df['quality'].value_counts()

In [ ]:
df.head()

<h1  style="text-align: center" class="list-group-item list-group-item-action active">5.2 Scaling</h1>

In [ ]:
# let's separate into training and testing set
X_train, X_test, y_train, y_test = train_test_split(df.drop('quality', axis=1),
                                                    df['quality'],
                                                    test_size=0.3,
                                                    random_state=0)

X_train.shape, X_test.shape

In [ ]:
# set up the scaler
scaler = StandardScaler()

# fit the scaler to the train set, it will learn the parameters
scaler.fit(X_train)

# transform train and test sets
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

<h1  style="text-align: center" class="list-group-item list-group-item-action active">6. Models</h1><a id = "6"></a>


In [ ]:
dct = DecisionTreeClassifier(random_state = 42)
svc = SVC(random_state = 42)
rf = RandomForestClassifier(random_state = 42)
logreg = LogisticRegression(random_state = 42)
knn = KNeighborsClassifier()
sgd = SGDClassifier()

In [ ]:
# Define the list classifiers
classifiers = [
    ("knn" , knn),
    ("rf" , rf),
    ("logreg" , logreg),
    ("svc", svc),
    ("sgd",sgd),
    ("dct",dct)
]

In [ ]:
# Iterate over the pre-defined list of classifiers
for clf_name, clf in classifiers:
    # Fit clf to the training set
    clf.fit(X_train, y_train)
    
    # Predict y_pred
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Evaluate clf's accuracy on the test set
    print('{:s} score : {:.3f}'.format(clf_name, acc))

In [ ]:
param_grid ={
    'n_estimators' : [50,100,200],
    'max_depth': (1,5,10),
    'min_samples_leaf': (1,5,10)
    }

In [ ]:
gridsearch = GridSearchCV(rf, param_grid=param_grid, scoring='accuracy', cv=5,n_jobs=6)

In [ ]:
%%capture
gridsearch.fit(X_train,y_train)

In [ ]:
gridsearch.best_params_

In [ ]:
rf = RandomForestClassifier(max_depth= 10, min_samples_leaf = 1, n_estimators = 100)

In [ ]:
rf.fit(X_train,y_train)

Let's predict!

In [ ]:
pred = rf.predict(X_test)

In [ ]:
print("Accuracy Score:",accuracy_score(pred,y_test))
print("classification Report:\n",classification_report(pred,y_test))

In [ ]:
cm = confusion_matrix(y_test, pred)

df1 = pd.DataFrame(columns=["Bad","Good"], index= ["Bad","Good"], data= cm )

f,ax = plt.subplots(figsize=(4,4))

sns.heatmap(df1, annot=True,cmap="Reds", fmt= '.0f',ax=ax,linewidths = 5, cbar = False)
plt.xlabel("Predicted Label")
plt.xticks(size = 12)
plt.yticks(size = 12, rotation = 0)
plt.ylabel("True Label")
plt.title("Confusion Matrix", size = 12)
plt.show()

If you find the notebook useful please give it a upvote 👍 


![](https://i0.wp.com/comicsandmemes.com/wp-content/uploads/blank-meme-template-139-crying-cat-thumbs-up-1.jpg?resize=650,400)
